# Smoke-Screen — a quantitative teardown 🔬
### The accruals long-short on real EDGAR data · the t-stat · the long leg beating the market · the short-side & decay caveats

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Accrual anomaly replicates?: Confirmed](https://img.shields.io/badge/Accrual_anomaly_replicates%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We show the accruals anomaly replicates significantly on SEC data.

> ⚠️ **Not investment advice.** EDGAR NetIncome/CFO/Assets + Yahoo annual returns, current S&P 500 members, ~2007–2025 (survivorship-biased, large-cap, short). Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (smoke_screen/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from smoke_screen import data, strategy as st
sig, fwd = data.fetch_panel()                  # cache-first (shared EDGAR pull)
h = st.quantile_hedge(sig, fwd, q=0.2, long_high=False)   # long low accruals


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | hedge +5.9%/yr, Sharpe 0.64, t ≈ 2.7, 72% hit |
| Tradability | **Fragile** | short-side cost, post-2000 fade, short sample |
| Replicates? | **Confirmed** | clean Sloan (1996) replication |

> 💡 *In plain words:* a real fundamental edge — cash beats accounting.

## 1 · The claim, steelmanned

- **H₁:** low-accruals firms out-earn high-accruals firms.
- **H₂:** it's significant.
- **H₃:** the long leg (cash-backed) beats the market.

## 2 · So what? — what rides on each

If all hold, quality-of-earnings is a confirmed, partly long-only-tradable edge. The short-side and decay caveats decide how much you keep.

## 3 · How we'd know — the protocol

Accruals = (NI − CFO)/Assets → annual long-low/short-high → the hedge's mean, Sharpe, t-stat, hit rate → the long leg vs the universe.

## 4 · The teardown

### 4.1 The hedge

In [2]:
display(h.round(3))
s=st.summary(h['hedge']); print(f"hedge: mean {s['mean']:+.2%}/yr  Sharpe {s['sharpe']:.2f}  t≈{s['tstat']:.1f}  hit {s['hit_rate']:.0%}  n={s['n']}")

,high,low,hedge
2008,0.352,0.451,0.099
2009,0.166,0.310,0.144
2010,-0.058,0.146,0.203
2011,0.338,0.210,-0.127
2012,0.516,0.476,-0.040
2013,0.159,0.221,0.062
2014,0.065,0.063,-0.002
2015,0.251,0.133,-0.118
2016,0.219,0.269,0.050
2017,-0.070,0.048,0.118


hedge: mean +5.89%/yr  Sharpe 0.64  t≈2.7  hit 72%  n=18


> 💡 *In plain words:* +5.9%/yr, t ≈ 2.7 over ~18 years — **H₁ and H₂ hold.** A genuine signal.

### 4.2 The long leg beats the market

In [3]:
print('low-accruals (long) : %+.2f%%' % (st.summary(h['low'])['mean']*100))
print('high-accruals (short): %+.2f%%' % (st.summary(h['high'])['mean']*100))
print('universe            : %+.2f%%' % (st.summary(st.market_annual(fwd).reindex(h.index))['mean']*100))

low-accruals (long) : +25.01%
high-accruals (short): +19.12%
universe            : +19.55%


> 💡 *In plain words:* cash-backed earners beat the market outright. **H₃ holds** — the edge isn't only on the short side.

### 4.3 The caveats that make it Fragile

1. **Short-side cost.** Shorting accrual-heavy names carries borrow and is the harder leg.
2. **Post-2000 fade.** Green-Hand-Soliman (2011) find the anomaly weakened markedly after ~2003 — a decay a ~18-year window can't reveal.
3. **Survivorship + large-cap.** A conservative test corner; a t of 2.7 here is, if anything, an under-statement.

## 5 · The verdict

H₁, H₂, H₃ all hold → Signal `REAL`, Tradability `FRAGILE`, replication `CONFIRMED`.

## 6 · Could you trade it?

The long-only version (own cash-backed earners) is clean and beats the market; the full long-short needs the costlier short and faces a known fade. A real edge, traded with care.

## 7 · Going further

Forks: (a) a longer Compustat history to see the post-2000 decay directly; (b) the percent-accruals / cash-flow variants (Hafzalla et al.); (c) combine with quality ([51 Blue-Chip](../../51-blue-chip/)). Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).